<a href="https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


Question shape: "which pages first?" is a ranking question with an observed yes/no label (views going down). So I use classifiers and rank pages by their predicted chance of "going down".

Methods:
1. Baseline rule from ML-07, frozen (same thresholds).
2. Logistic regression (the readable model).
3. Random forest (stronger but harder to read, to see if the extra complexity earns its place).

I kept both models simple: one fixed setting each, no tuning, seeds fixed at 42.

Inputs: observed signals only (traffic totals, position, click rate, page age, days since update, page type and intent). Banned as inputs: trend_direction, trend_pct, the last-30-day and previous-30-day columns (they build the label), and the IDs. Missing values are filled with the training-fold median plus a "was missing" flag. avg_position = 0 means "no data", so it is treated as missing.

Careful words: results are observed on this starter file, directional, and for decision-support only.

In [1]:
import os, sys, subprocess, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# In Colab: download the repo once. On your computer: go to the repo folder.
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("Flyrank-MLE-intern"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ysl21292/Flyrank-MLE-intern.git"], check=True)
    os.chdir("Flyrank-MLE-intern")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["declined"] = (df.trend_direction == "down").astype(int)
y = df.declined.values

# Features: observed signals only. These columns can never be inputs (they build the label or are IDs).
NEVER = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
         "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
         "content_id", "client_id"]
num_cols = ["search_volume", "competition", "cpc", "word_count", "char_count", "days_with_impressions",
            "days_with_sessions", "content_age_days", "days_since_last_update", "ctr", "avg_position",
            "engagement_rate", "scroll_rate", "ai_traffic_pct"]
X = df[num_cols].copy()
X["avg_position"] = X.avg_position.replace(0, np.nan)          # 0 means "no data"
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    X["log_" + c] = np.log1p(df[c])                             # traffic is heavy-tailed
X = pd.concat([X, pd.get_dummies(df[["content_type", "main_intent", "competition_level"]].fillna("unknown"), dtype=float)], axis=1)
assert not any(c in X.columns for c in NEVER)
print(f"{X.shape[1]} input columns, none from the banned list. Seeds fixed at 42.")

# Missing values: fill with the median inside each training fold, and add a "was missing" flag column.
def make_model(kind):
    fill = SimpleImputer(strategy="median", add_indicator=True)
    if kind == "logistic":
        return make_pipeline(fill, StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    return make_pipeline(fill, RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42, n_jobs=-1))

# The baseline rule from ML-07, frozen (same rule, same thresholds).
flagged = (df.content_age_days < 180) & (df.days_since_last_update >= 90) & (df.impressions_90d >= 100) & (df.avg_position > 0)
rule_score = np.where(flagged, df.days_since_last_update, 0)

30 input columns, none from the banned list. Seeds fixed at 42.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I use 5-fold cross-validation grouped by client. Each fold holds out whole clients, so the model is always tested on clients it never saw (the printout shows 0 clients in both train and test). A random split would put a client's pages on both sides and make the score look better than it is.

Honest notes: one fold is a single very large client, and the folds have different base rates (0.38 to 0.65). So I report the mean over folds plus the worst and best fold. The ML-07 rule was designed by looking at the whole file, so it had a head start over the models.

In [2]:
from sklearn.model_selection import GroupKFold

# 5 folds. Whole clients go to test, so the model is always tested on clients it never trained on.
folds = list(GroupKFold(n_splits=5).split(X, y, groups=df.client_id))
for i, (tr, te) in enumerate(folds):
    both = set(df.client_id.iloc[tr]) & set(df.client_id.iloc[te])
    print(f"fold {i}: test pages {len(te):,} | test clients {df.client_id.iloc[te].nunique()} | "
          f"base rate {y[te].mean():.2f} | clients in both train and test: {len(both)}")

fold 0: test pages 7,008 | test clients 1 | base rate 0.49 | clients in both train and test: 0
fold 1: test pages 5,731 | test clients 7 | base rate 0.65 | clients in both train and test: 0
fold 2: test pages 5,753 | test clients 8 | base rate 0.38 | clients in both train and test: 0
fold 3: test pages 5,755 | test clients 8 | base rate 0.62 | clients in both train and test: 0
fold 4: test pages 5,753 | test clients 8 | base rate 0.58 | clients in both train and test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


Same data, same folds, same metrics for all three.


What I see (directional):
- Both models beat random picking and the rule on AUC. Logistic regression is ahead of the rule on average at P@20, P@50 and P@100.
- At P@50 the gap is small next to the swing between folds. The rule beats logistic regression in folds 3 and 4, so I cannot call this a clear win.
- The rule only flags pages in some clients. In fold 2 it flagged just 7 pages, so it was close to random there (0.44). The models rank every page.
- The random forest has the best P@20 and AUC but the least stable P@50 (0.56 to 0.90), so the extra complexity did not clearly earn its place. Logistic regression is the steadier pick.

In [3]:
from sklearn.metrics import roc_auc_score

tiebreak = np.random.default_rng(42).random(len(df))      # random tie-break, not file order

def precision_at_k(score, yt, tb, k):
    order = np.lexsort((tb, -np.asarray(score)))[:k]
    return yt[order].mean()

oof = {"logistic": np.zeros(len(df)), "forest": np.zeros(len(df))}
rows = []
for i, (tr, te) in enumerate(folds):
    scores = {"Baseline rule": rule_score[te]}
    for name, kind in [("Logistic regression", "logistic"), ("Random forest", "forest")]:
        m = make_model(kind).fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        oof[kind][te] = p
        scores[name] = p
    for name, s in scores.items():
        rows.append({"fold": i, "method": name, "base_rate": y[te].mean(),
                     "P@20": precision_at_k(s, y[te], tiebreak[te], 20),
                     "P@50": precision_at_k(s, y[te], tiebreak[te], 50),
                     "P@100": precision_at_k(s, y[te], tiebreak[te], 100),
                     "AUC": roc_auc_score(y[te], s)})
res = pd.DataFrame(rows)

summary = res.groupby("method").agg(P20=("P@20", "mean"), P50=("P@50", "mean"), P50_worst=("P@50", "min"),
                                    P50_best=("P@50", "max"), P100=("P@100", "mean"), AUC=("AUC", "mean")).round(2)
summary = summary.loc[["Baseline rule", "Logistic regression", "Random forest"]]
print(f"Base rate (random picking): {res.groupby('fold').base_rate.first().mean():.2f}\n")
print("Mean over 5 folds (each fold = unseen clients):")
print(summary.to_string())
print("\nPrecision@50 per fold:")
print(res.pivot(index="fold", columns="method", values="P@50").round(2).to_string())
print("\nBaseline-flagged pages per test fold:", [int(flagged.iloc[te].sum()) for _, te in folds])

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(summary.reset_index().to_dict(orient="records"), f, indent=2)

Base rate (random picking): 0.54

Mean over 5 folds (each fold = unseen clients):
                      P20   P50  P50_worst  P50_best  P100   AUC
method                                                          
Baseline rule        0.68  0.74       0.44      0.92  0.71  0.52
Logistic regression  0.76  0.80       0.62      0.88  0.81  0.66
Random forest        0.80  0.74       0.56      0.90  0.77  0.67

Precision@50 per fold:
method  Baseline rule  Logistic regression  Random forest
fold                                                     
0                0.74                 0.82           0.90
1                0.70                 0.86           0.90
2                0.44                 0.62           0.74
3                0.88                 0.82           0.60
4                0.92                 0.88           0.56

Baseline-flagged pages per test fold: [610, 829, 7, 411, 87]


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


What the model leans on (permutation importance on unseen clients): log_clicks_90d by far (AUC drops by 0.16 when shuffled), then log_impressions_90d (0.05), log_sessions_90d (0.03), word_count (0.02) and avg_position (0.02).

Sanity check: the top features are 90-day traffic totals. Those totals contain the same 60 days that define my label (last 30 days vs the 30 days before). So part of the signal may be overlap and not a real early warning. This is the main weakness of the starter label. A future-window label from the warehouse (features from the past, outcome from the next 30 days) would fix it.

Where it is wrong:
- By content type, the model over-predicts decline for comparison articles (predicted 0.68, actual 0.57, n=697). Feedly and keyword articles are close.
- By position tier, the average predicted and actual rates are close, but averages can hide ranking mistakes.
- The three wrong picks are all keyword articles with many views (3,881 to 19,360), a position of about 6 to 9, and almost no clicks (CTR 0.00 to 0.01), yet they did not decline. A page with big impressions and almost no clicks looks like it is leaking views, but these held steady. I think a 30-day change is noisy for such pages, but I have not tested that.

Claims: this ranks pages for a human to check (decision-support). It does not show that updating a page helps.

In [4]:
from sklearn.inspection import permutation_importance

# What does the simple model lean on? Shuffle one column at a time on unseen clients and see how much AUC drops.
tr, te = folds[1]
lr = make_model("logistic").fit(X.iloc[tr], y[tr])
pi = permutation_importance(lr, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=42)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
print("Top 5 features (AUC drop when shuffled, fold 1 unseen clients):")
print(imp.head(5).round(3).to_string())

# Where is the logistic model wrong? Predicted vs actual declining rate by content type and position.
chk = pd.DataFrame({"p": oof["logistic"], "y": y, "content_type": df.content_type, "position_tier": df.position_tier})
for col in ["content_type", "position_tier"]:
    g = chk.groupby(col).agg(n=("y", "size"), actual=("y", "mean"), predicted=("p", "mean")).round(2)
    print(f"\nBy {col} (n = pages):")
    print(g[g.n >= 100].to_string())

# Three concrete wrong picks: pages the model put at the top of its fold's list that did NOT decline.
wrong = []
for tr_i, te_i in folds:
    p = oof["logistic"][te_i]
    top = te_i[np.lexsort((tiebreak[te_i], -p))[:50]]
    wrong += [i for i in top if y[i] == 0]
wrong = sorted(wrong, key=lambda i: -oof["logistic"][i])[:3]
print("\nThree wrong picks (model score high, page did not decline):")
print(df.loc[wrong, ["content_type", "impressions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update"]].to_string(index=False))

Top 5 features (AUC drop when shuffled, fold 1 unseen clients):
log_clicks_90d         0.164
log_impressions_90d    0.052
log_sessions_90d       0.033
word_count             0.024
avg_position           0.020

By content_type (n = pages):
                        n  actual  predicted
content_type                                
comparison article    697    0.57       0.68
feedly article       2096    0.29       0.29
keyword article     27207    0.56       0.55

By position_tier (n = pages):
                   n  actual  predicted
position_tier                          
deep            1319    0.34       0.35
page_1         11814    0.57       0.57
page_3_5        7242    0.56       0.53
striking        7304    0.61       0.58
top_3           2321    0.24       0.26

Three wrong picks (model score high, page did not decline):
   content_type  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update
keyword article            15101           5.7 0.00               421 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.